# Country-level temperature trends

`get_era5_country_temperature()` resolves a country boundary from
[Natural Earth](https://www.naturalearthdata.com/) data, downloads monthly ERA5
temperature, and returns cosine-latitude area-weighted national averages. One
call replaces the manual workflow of finding boundaries, writing GeoJSON,
calling `era5ify_geojson()`, and aggregating grid cells.

We pull monthly mean, max, and min temperatures for four European and four
South Asian countries.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import calendar

import varunayan as v

## Download data

We pass the shorthands `"mean"`, `"max"`, and `"min"` for the three ERA5
temperature variables.

In [ ]:
countries = [
    "France", "Germany", "Spain", "Norway",
    "India", "Pakistan", "Bangladesh", "Nepal",
]

frames = []
for name in countries:
    df = v.get_era5_country_temperature(
        country=name,
        start_date="2024-01-01",
        end_date="2024-12-31",
        variables=["mean", "max", "min"],
    )
    df["country"] = name
    frames.append(df)

temps = pd.concat(frames, ignore_index=True)
temps.head()

In [ ]:
region_map = {
    "France": "Europe", "Germany": "Europe",
    "Spain": "Europe", "Norway": "Europe",
    "India": "South Asia", "Pakistan": "South Asia",
    "Bangladesh": "South Asia", "Nepal": "South Asia",
}

temps["region"] = temps["country"].map(region_map)
temps["month_label"] = temps["month"].apply(lambda m: calendar.month_abbr[m])

country_colors = {
    "France": "#2166AC", "Germany": "#4393C3",
    "Spain": "#92C5DE", "Norway": "#053061",
    "India": "#B2182B", "Pakistan": "#D6604D",
    "Bangladesh": "#F4A582", "Nepal": "#67001F",
}

## Seasonal heatmap

In [ ]:
europe = [c for c in countries if region_map[c] == "Europe"]
sasian = [c for c in countries if region_map[c] == "South Asia"]
ordered = europe + sasian

pivot = temps.pivot_table(
    index="country", columns="month", values="temperature_mean",
).reindex(ordered)

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(pivot.values, aspect="auto", cmap="RdYlBu_r")
ax.set_xticks(range(12))
ax.set_xticklabels([calendar.month_abbr[m] for m in range(1, 13)])
ax.set_yticks(range(len(ordered)))
ax.set_yticklabels(ordered)
ax.axhline(len(europe) - 0.5, color="black", lw=1)
fig.colorbar(im, ax=ax, label="Mean temp (\u00b0C)", shrink=0.7)
ax.set_title("Mean monthly temperature by country (2024)")
plt.tight_layout()
plt.show()

## Monthly trend lines with min/max ribbon

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

for ax, (region, group) in zip(axes, temps.groupby("region")):
    for name, sub in group.groupby("country"):
        color = country_colors[name]
        ax.fill_between(
            sub["month"], sub["temperature_min"], sub["temperature_max"],
            alpha=0.12, color=color,
        )
        ax.plot(sub["month"], sub["temperature_mean"], color=color, lw=1.2,
                marker="o", ms=3, label=name)
    ax.set_ylabel("Temperature (\u00b0C)")
    ax.set_title(region)
    ax.legend(loc="best", fontsize=9)
    ax.grid(True, alpha=0.3)

axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels([calendar.month_abbr[m] for m in range(1, 13)])
fig.suptitle("Monthly temperature with min-max range (2024)", fontsize=13)
plt.tight_layout()
plt.show()

## Diurnal temperature range

The gap between daily max and min (DTR) is wider in continental and arid
climates than in humid tropical ones.

In [ ]:
temps["dtr"] = temps["temperature_max"] - temps["temperature_min"]

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

for ax, (region, group) in zip(axes, temps.groupby("region")):
    for name, sub in group.groupby("country"):
        ax.plot(sub["month"], sub["dtr"], color=country_colors[name], lw=1.2,
                marker="o", ms=3, label=name)
    ax.set_ylabel("DTR (\u00b0C)")
    ax.set_title(region)
    ax.legend(loc="best", fontsize=9)
    ax.grid(True, alpha=0.3)

axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels([calendar.month_abbr[m] for m in range(1, 13)])
fig.suptitle("Monthly diurnal temperature range (2024)", fontsize=13)
plt.tight_layout()
plt.show()

## Regional comparison

All eight countries on a single axis, with line style separating the two
regions.

In [ ]:
linestyles = {"Europe": "-", "South Asia": "--"}

fig, ax = plt.subplots(figsize=(10, 5))

for name, sub in temps.groupby("country"):
    region = region_map[name]
    ax.plot(sub["month"], sub["temperature_mean"],
            color=country_colors[name], ls=linestyles[region],
            lw=1, marker="o", ms=3, label=name)

ax.set_xticks(range(1, 13))
ax.set_xticklabels([calendar.month_abbr[m] for m in range(1, 13)])
ax.set_ylabel("Mean temperature (\u00b0C)")
ax.set_title("Europe vs South Asia: monthly temperature (2024)")
ax.legend(ncol=2, fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()